In [2]:
!pip install transformers

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [3]:
device ="cpu"

In [4]:
import json
import torch
import numpy as np
from typing import Optional
import pandas as pd
import copy
import os
import re
#import ipdb

In [5]:
import os
import pickle
from transformers import (
    BertTokenizer,
    BertForMaskedLM,
    VisualBertForPreTraining,
    RobertaTokenizer,
    RobertaForMaskedLM,
    LxmertTokenizer,
    LxmertForPreTraining,
    ViltProcessor,
    ViltForMaskedLM,
    FlavaForPreTraining,
    AutoProcessor
)

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Calcolo della pseudo-log-likelihood

In [7]:
class Pseudo_log_likelihood:

    def __init__(self, tokenizer, model: torch.nn.Module, model_name, sentence):
        self.tokenizer = tokenizer
        self.model = model
        self.model_name = model_name
        self.sentence = sentence
        self.cached_plls: Optional[np.ndarray] = None
        self.sent, self.score = self.pseudo_log_likelihood(self.sentence, self.model_name)


    def get_masked_seq(self, tokens, tokenized_words, tokenizer, model_name):
        """
        La maschera si applica all'intera frase parola per parola.
        Se una parola è tokenizzata in più token, si applica una maschera multipla.
        """
        nr_masks = [len(elm) for elm in tokenized_words]

        masked_seq = []
        i = 1 #start with 0 because we want to predict for the first element
        j = 1
        while i < len(tokens) -1: #don't subtract 1 from len(tokens) to include the last element
            curr_masked_seq = []
            if nr_masks[j] == 1:
                curr_masked_seq = [tokens[ind] if ind != i else tokenizer.mask_token for ind in range(len(tokens))]
                masked_seq.append(curr_masked_seq)
                i += 1
                j += 1

            else:
                for k in range(nr_masks[j]):
                    curr_nr_masks = nr_masks[j] - k
                    curr_masked_seq = [tokens[ind] if (ind < i+k or ind >= i+k+curr_nr_masks) else tokenizer.mask_token for ind in range(len(tokens))]
                    masked_seq.append(curr_masked_seq)
                i += nr_masks[j]
                j += 1

        return masked_seq

    def prepare_input(self, sentence, model_name):

        tokens = self.tokenizer.tokenize(sentence)
        tokens = [self.tokenizer.cls_token] + tokens + [self.tokenizer.sep_token]
        #print(f"Tokens: {tokens}")
        tokenized_words = []

        """Dividi le parole in token sulla base del tipo di tokenizzazione"""

        if model_name in ["BERT", "VisualBERT", "ViLT", "FLAVA", "LXMERT"]:
            for ind, tok in enumerate(tokens):
                if not re.match("#", tok):
                    curr_word = [tok]
                else:
                    curr_word.append(tok)

                #end a word
                if ind == len(tokens) - 1:
                    tokenized_words.append(curr_word)
                else:
                    if not re.match("#", tokens[ind+1]):
                        tokenized_words.append(curr_word)

        elif model_name == 'RoBERTa':
            for ind, tok in enumerate(tokens):
                #special cases
                if tok in [self.tokenizer.cls_token, self.tokenizer.sep_token, "."] or ind == 1: #CLS, SEP & first word
                    curr_word = [tok]
                elif re.match("Ġ", tok): # accounting for special tokens
                    curr_word = [tok]
                else:
                    curr_word.append(tok)

                #end a word
                if tok in [self.tokenizer.cls_token, self.tokenizer.sep_token, "."]:
                    tokenized_words.append(curr_word)
                elif ind < len(tokens):
                    if re.match(r"Ġ|\.", tokens[ind+1]): #end word in case final period is next or new word is next
                        tokenized_words.append(curr_word)

        else:
            raise NotImplementedError

        #print(f"Tokenized words: {tokenized_words}")

        masked_seq = self.get_masked_seq(tokens, tokenized_words, self.tokenizer, self.model_name)
        #print(f"Masked seq: {masked_seq}")

        return tokens, masked_seq

    def pseudo_log_likelihood(self, sentence, model_name):
        mask_token_id = self.tokenizer.mask_token_id
        max_len = 20

        tokens, masked_seq = self.prepare_input(sentence, model_name)
        nr_tokens_to_predict = len(tokens) - 2 #because of CLS & SEP
        list_of_sents = [sentence] * nr_tokens_to_predict

        encoded_inputs = self.tokenizer(list_of_sents, padding="max_length", max_length=20)

        for i in range(len(masked_seq)):
            for j in range(len(masked_seq[i])):
                if masked_seq[i][j] == self.tokenizer.mask_token:
                    encoded_inputs["input_ids"][i][j] = self.tokenizer.mask_token_id

        """
           Il modello di pretraining di FLAVA
           nel compito di mlm con in input solo testo
           richiede che sia specificato che gli input_ids sono mascherati
        """

        if isinstance(self.model, FlavaForPreTraining):

            input_ids_masked = torch.tensor(encoded_inputs["input_ids"])
            attention_mask = torch.tensor(encoded_inputs["attention_mask"])

        else:
            input_ids = torch.tensor(encoded_inputs["input_ids"])
            attention_mask = torch.tensor(encoded_inputs["attention_mask"])

        #II torch.Size([7, 20]) AM torch.Size([7, 20]) TTI torch.Size([7, 20])
        #print("II", input_ids.shape, "AM", attention_mask.shape)#, "TTI", token_type_ids.shape)

        # calcola la pseudo-log-likelihood
        self.model.eval()
        with torch.no_grad():

            """
               Ogni modello multimodale integra la parte visuale in maniera diversa:
               LXMERT utilizzando features esterne estratte con una rete R-CNN
               ViLT estraendo direttamente i pixel_values dall'immagine
               FLAVA non richiede una parte visuale nella predizione delle parole mascherate
            """
            if isinstance(self.model, LxmertForPreTraining):

                visual_features = torch.zeros(1, 36, 2048) #([1, 36, 2048])
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    visual_feats= visual_features,
                    visual_attention_mask = torch.ones(visual_features.shape[:-1], dtype=torch.long),
                    visual_pos=torch.zeros(1, 36, 4) #torch.Size([1, 36, 4])
                    )

            elif isinstance(self.model, ViltForMaskedLM):

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    pixel_values = torch.zeros((len(encoded_inputs.input_ids)), 3, 384, 512) #default dimensions (batch_size, num_channels, height, width)
                )

            elif isinstance(self.model, FlavaForPreTraining):

                outputs = self.model(
                    input_ids_masked = input_ids_masked,
                    attention_mask=attention_mask,
                    )

            else:

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                    )

        log_probs_fillers = []
        all_log_probs=[]
        predict_token = tokens[1:-1]

        """
            Anche i logits possono essere denotati in maniera diversa.
            In particolare, in FLAVA possiamo utilizzare i logits relativi alla predizione del testo mascherato
            sulla base del solo input testuale
            oppure sulla base di input testuale e visuale
        """

        if isinstance(self.model, (LxmertForPreTraining, VisualBertForPreTraining)):
            logits = outputs.prediction_logits

        elif isinstance(self.model, FlavaForPreTraining):
            #logits = outputs.mmm_text_logits
            logits = outputs.mlm_logits

        else:
            logits = outputs.logits

        for batch_elem, token, index in zip(range(len(logits)), predict_token, range(1, len(predict_token) + 1)): #no need for CLS & SEP
            all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])
            log_probs_fillers.append(all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item())
            #print(f"{self.tokenizer.convert_tokens_to_ids(token)} | {token} | {all_log_probs[self.tokenizer.convert_tokens_to_ids(token)].item()}\n")

        sentence_score = sum(log_probs_fillers)
        print(f" {sentence} | {sentence_score}\n")
        #self.cached_plls = np.array([sentence_score])

        return sentence, sentence_score

#MAIN

In [8]:
import os
import pickle
import torch as t
import numpy as np
from typing import Optional
from torch.nn import functional as F
import pandas as pd
import os
import glob
import json

In [9]:



def main():

    force = True
    result_plls = {}
    print(f"\nforce={force}\n")
    if os.path.exists("/content/"):
        data_dir = "/content/drive/MyDrive/data/esempio"
    else:
        data_dir = "data"
    models = {
        "BERT": (
            BertTokenizer.from_pretrained("bert-base-uncased"),
            BertForMaskedLM.from_pretrained("bert-base-uncased")
        ),
        "RoBERTa": (
            RobertaTokenizer.from_pretrained("roberta-base"),
            RobertaForMaskedLM.from_pretrained("roberta-base")
        ),
        "VisualBERT": (
            BertTokenizer.from_pretrained("bert-base-uncased"),
            VisualBertForPreTraining.from_pretrained("uclanlp/visualbert-nlvr2-coco-pre")
        ),
        "ViLT": (
            ViltProcessor.from_pretrained("dandelin/vilt-b32-mlm").tokenizer,
            ViltForMaskedLM.from_pretrained("dandelin/vilt-b32-mlm")
        ),
        "FLAVA": (
            AutoProcessor.from_pretrained("facebook/flava-full").tokenizer,
            FlavaForPreTraining.from_pretrained("facebook/flava-full")
        ),
        "LXMERT": (
            LxmertTokenizer.from_pretrained("unc-nlp/lxmert-base-uncased"),
            LxmertForPreTraining.from_pretrained("unc-nlp/lxmert-base-uncased")
        ),
    }

    txt_files = list(glob.glob(f"{data_dir}/*.txt"))
    global_results = {}
    """save the results with the file name"""
    for filename in txt_files:
        print(filename)
        if force or (not os.path.exists(f"{filename}_result_plls.pkl")):
            for model_name, (tokenizer, model) in list(models.items()):
                results = {}
                global_results[filename] = results
                print("File Name:", filename.split("/")[-1])
                out_file_name = os.path.join(
                    "txt_results", f"{model_name}_{os.path.basename(filename)}_sentence.txt"
                )
                data = pd.read_csv(filename, sep="\t", header=None)
                results["len(data)"] = len(data)
                model_results = {}
                results[model_name] = model_results
                result = {
                        "sentences": [],
                        "sent_ppls": [],
                        "num_words": [],
                        "num_tokens": [],
                        }
                print(f"\n\nEvaluating: {model_name}")
                for idx, row in enumerate(data.itertuples()):
                    sentence =row[2]
                    pll = Pseudo_log_likelihood(
                        tokenizer, model, model_name, sentence
                    )
                    tokens = tokenizer.tokenize(sentence)
                    num_tokens = len(tokens)
                    words = row[2].split()
                    num_words = len(words)
                    result["sentences"].append(pll.sent)
                    result["sent_ppls"].append(pll.score)
                    result["num_words"].append(num_words)
                    result["num_tokens"].append(num_tokens)
        else:
            with open(f"{filename}_result_plls.pkl", "rb") as file:
                pll.score = pickle.load(file)
            # uses pandas to save the results in a csv file
        pd.DataFrame(result).to_csv(
            out_file_name, sep="\t", header=None, index=None
        )


if __name__ == '__main__':
    main()


force=True



Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['cls.seq_relationship.weight', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
`text_config_dict` is provided which will be used to initialize `FlavaTextConfig`. The value `text_config["id2label"]` will be overriden.
`multimodal_config_dict` is provided which will be used to initialize `FlavaMultimodalConfig`. The value `multimodal_config["id2label"]` will be overriden.
`image_codebook_config_dict` is provided which will be used to initialize

/content/drive/MyDrive/data/esempio/demo.txt
File Name: demo.txt


Evaluating: BERT


<ipython-input-7-d05099f50ddb>:188: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


 The cop is arresting the criminal . | -21.443622294813395

 The criminal is arresting the cop . | -24.386674612760544

 The babysitter is scolding the child . | -41.90585809899494

 The child is scolding the babysitter . | -42.247277666814625

 The doctor is using a stethoscope on the patient . | -32.66431935550645

 The patient is using a stethoscope on the doctor . | -48.165163803845644

File Name: demo.txt


Evaluating: RoBERTa


...

 The cop is arresting the criminal . | -38.837265104055405

 The criminal is arresting the cop . | -38.51042675226927

 The babysitter is scolding the child . | -61.414930794388056

 The child is scolding the babysitter . | -59.37989580631256

 The doctor is using a stethoscope on the patient . | -9.877188239246607

 The patient is using a stethoscope on the doctor . | -19.277624674141407

File Name: demo.txt


Evaluating: FLAVA


/usr/local/lib/python3.10/dist-packages/transformers/modeling_utils.py:884: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
<ipython-input-7-d05099f50ddb>:188: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


 The cop is arresting the criminal . | -14.424903929233551

 The criminal is arresting the cop . | -20.22563938051462

 The babysitter is scolding the child . | -25.233239863067865

 The child is scolding the babysitter . | -29.55945379845798

 The doctor is using a stethoscope on the patient . | -32.60840251063928

 The patient is using a stethoscope on the doctor . | -41.57473839819431

File Name: demo.txt


Evaluating: LXMERT


<ipython-input-7-d05099f50ddb>:188: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  all_log_probs = torch.nn.functional.log_softmax(logits[batch_elem, index])


 The cop is arresting the criminal . | -39.808327466249466

 The criminal is arresting the cop . | -43.7295800447464

 The babysitter is scolding the child . | -51.301841892302036

 The child is scolding the babysitter . | -50.23145787976682

 The doctor is using a stethoscope on the patient . | -51.90620003268123

 The patient is using a stethoscope on the doctor . | -52.16904238983989

